# Module 1 — Introduction to Statistics (Python Practical)

**Environment:** Google Colab · **Libraries:** pandas, numpy, matplotlib, seaborn

## Learning outcomes
By the end, you should be able to:
- create and inspect a small business dataset;
- identify population, sample, parameter, statistic and census;
- calculate and interpret descriptive summaries;
- distinguish descriptive analysis from inference, prediction and causal questions;
- use basic statistical checks to monitor a data pipeline.

**Working rule:** Before running each cell, predict what it will show. After running it, write a one- or two-sentence interpretation in business language.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
np.random.seed(42)

print("Libraries loaded successfully")


## 1. Create a realistic synthetic order dataset

Each row represents one order. The full generated table will be treated as our **finite study population** for this exercise. In a real project, you would first verify that the source table covers every eligible real-world order.

In [ ]:
n = 1200
cities = np.random.choice(["Bengaluru", "Delhi", "Mumbai", "Pune"], size=n,
                          p=[0.35, 0.25, 0.25, 0.15])
weekday = np.random.choice(["Weekday", "Weekend"], size=n, p=[0.72, 0.28])
order_value = np.random.gamma(shape=2.5, scale=450, size=n) + 100
delivery_minutes = np.random.normal(38, 10, size=n)
delivery_minutes += np.where(cities == "Bengaluru", 4, 0)
delivery_minutes += np.where(weekday == "Weekend", 3, 0)
delivery_minutes = np.clip(delivery_minutes, 12, 100).round(1)
late = (delivery_minutes > 50).astype(int)
status = np.where(np.random.rand(n) < 0.025, "Cancelled", "Completed")

orders = pd.DataFrame({
    "order_id": [f"ORD{i:05d}" for i in range(1, n+1)],
    "city": cities,
    "day_type": weekday,
    "order_value_inr": order_value.round(2),
    "delivery_minutes": delivery_minutes,
    "is_late": late,
    "status": status
})

orders.head()


In [ ]:
print("Rows:", len(orders))
print("Columns:", orders.columns.tolist())
print("Shape:", orders.shape)
print("\nData types:")
print(orders.dtypes)
print("\nMissing values:")
print(orders.isna().sum())


### Your interpretation
Write down:
1. What is one observation/row?
2. What is the unit of analysis?
3. What does each column represent?
4. What would make this generated table different from a real production dataset?


## 2. Population, sample, parameter and statistic

In [ ]:
# For this exercise, all 1,200 generated rows are our finite study population.
population = orders.copy()

# Draw a simple random sample of 150 orders.
sample = population.sample(n=150, random_state=7)

population_mean_delivery = population["delivery_minutes"].mean()  # parameter for this finite population
sample_mean_delivery = sample["delivery_minutes"].mean()            # statistic
population_late_rate = population["is_late"].mean()                 # parameter
sample_late_rate = sample["is_late"].mean()                          # statistic

print(f"Population size: {len(population)}")
print(f"Sample size: {len(sample)}")
print(f"Population mean delivery: {population_mean_delivery:.2f} minutes")
print(f"Sample mean delivery: {sample_mean_delivery:.2f} minutes")
print(f"Population late-order proportion: {population_late_rate:.3f}")
print(f"Sample late-order proportion: {sample_late_rate:.3f}")


**Interpretation prompt:** The population values above are known only because this is a simulation and we generated the entire finite population. In real work, the target population parameter is usually unknown. The sample statistic may differ from it because another random sample would contain different orders.

In [ ]:
# Repeat the sampling to see sample-to-sample variation.
sample_means = []
for seed in range(100):
    s = population.sample(n=150, random_state=seed)
    sample_means.append(s["delivery_minutes"].mean())

print("Range of 100 sample means:",
      round(min(sample_means), 2), "to", round(max(sample_means), 2))
print("True finite-population mean:", round(population_mean_delivery, 2))

plt.figure(figsize=(8, 4))
sns.histplot(sample_means, bins=15, kde=True)
plt.axvline(population_mean_delivery, linestyle="--", label="Population mean")
plt.title("Variation in sample means across 100 samples")
plt.xlabel("Sample mean delivery time (minutes)")
plt.legend()
plt.show()


### Exercise 1 — change the sample size
Run the sampling experiment with sample sizes 30, 100, 300 and 800. Record the range of sample means for each size. What happens as sample size grows? Does this prove that a biased sample becomes representative? Explain.

In [ ]:
# TODO: complete this experiment.
sample_sizes = [30, 100, 300, 800]
results = []

for size in sample_sizes:
    means = []
    for seed in range(100):
        s = population.sample(n=size, random_state=seed)
        means.append(s["delivery_minutes"].mean())
    results.append({
        "sample_size": size,
        "minimum_sample_mean": min(means),
        "maximum_sample_mean": max(means),
        "range": max(means) - min(means)
    })

pd.DataFrame(results)


## 3. Census versus sample

In [ ]:
# A census attempts to include every eligible unit in the defined population.
census_table = population.copy()

# Example of a sample-based manual quality audit.
audit_sample = population.sample(n=100, random_state=21)

print("Census rows:", len(census_table))
print("Audit sample rows:", len(audit_sample))


**Think critically:** The census table has every row in our generated population, but could a real warehouse table still be incomplete? Yes: events may fail ingestion, arrive late, be duplicated, or be excluded by incorrect filters. Census does not mean error-free.

## 4. Descriptive statistics

In [ ]:
numeric_summary = orders[["order_value_inr", "delivery_minutes"]].describe()
numeric_summary


In [ ]:
print("Mean order value:", orders["order_value_inr"].mean())
print("Median order value:", orders["order_value_inr"].median())
print("Mean delivery time:", orders["delivery_minutes"].mean())
print("Median delivery time:", orders["delivery_minutes"].median())
print("90th percentile delivery time:", orders["delivery_minutes"].quantile(0.90))
print("Standard deviation of delivery time:", orders["delivery_minutes"].std())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(orders["order_value_inr"], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Order value distribution")
sns.histplot(orders["delivery_minutes"], bins=25, kde=True, ax=axes[1])
axes[1].set_title("Delivery time distribution")
plt.tight_layout()
plt.show()


### Interpretation task
- Why might mean order value be higher than median order value?
- Which is more useful for describing a typical order value here, mean or median? Defend your choice.
- What does the 90th percentile delivery time mean in plain language?
- Why is standard deviation useful alongside the mean?


## 5. Descriptive analysis by segment

In [ ]:
city_summary = orders.groupby("city").agg(
    orders=("order_id", "count"),
    mean_delivery=("delivery_minutes", "mean"),
    median_delivery=("delivery_minutes", "median"),
    late_rate=("is_late", "mean"),
    mean_order_value=("order_value_inr", "mean")
).sort_values("late_rate", ascending=False)

city_summary


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=city_summary.reset_index(), x="city", y="late_rate")
plt.title("Late-order proportion by city")
plt.ylabel("Late-order proportion")
plt.xlabel("City")
plt.ylim(0, 1)
plt.show()


**Business interpretation:** Identify the city with the highest observed late-order proportion. Before recommending action, check sample sizes, time period, delivery-zone mix, order mix and whether the differences are practically important. This table is descriptive; it does not prove why the differences exist.

## 6. Classify the analysis question

For each question, label it as **descriptive, exploratory/diagnostic, inferential, predictive or causal**. Some questions can involve more than one type; explain your reasoning.

1. What was the median delivery time last month?
2. Is the late-delivery rate in a probability sample consistent with a 5% population target?
3. Which future orders are most likely to be late?
4. Did introducing a new dispatch policy reduce delivery time?
5. Which zones show an unusual rise in missing delivery timestamps?


## 7. Data Engineering: statistical monitoring checks

In [ ]:
# Daily row-count and data-quality style monitoring.
# We create an artificial run label to demonstrate monitoring summaries.
monitor = orders.copy()
monitor["pipeline_run"] = np.random.choice(
    ["run_1", "run_2", "run_3", "run_4"], size=len(monitor)
)

run_health = monitor.groupby("pipeline_run").agg(
    rows=("order_id", "count"),
    null_delivery_rate=("delivery_minutes", lambda x: x.isna().mean()),
    duplicate_order_ids=("order_id", lambda x: x.duplicated().sum()),
    late_rate=("is_late", "mean"),
    average_delivery=("delivery_minutes", "mean")
)
run_health


### Inject a data-quality problem
The next cell deliberately creates missing delivery times and duplicate IDs. Observe how monitoring changes. A statistical anomaly should trigger investigation, not automatic deletion or a blind correction.

In [ ]:
bad_run = orders.copy()
bad_run.loc[bad_run.sample(frac=0.08, random_state=10).index, "delivery_minutes"] = np.nan
bad_run = pd.concat([bad_run, bad_run.iloc[:12]], ignore_index=True)

quality_report = pd.DataFrame({
    "metric": ["row_count", "missing_delivery_rate", "duplicate_order_id_count"],
    "value": [
        len(bad_run),
        bad_run["delivery_minutes"].isna().mean(),
        bad_run["order_id"].duplicated().sum()
    ]
})
quality_report


### Engineering interpretation
Write an incident note containing:
- what metric changed;
- the observed value;
- the expected baseline or threshold (if known);
- the likely impact on downstream reporting;
- the first three checks you would perform;
- why you should preserve raw data while investigating.


## 8. Data-driven decision making

In [ ]:
# Build a compact decision table from observed data.
decision_view = orders.groupby("city").agg(
    order_count=("order_id", "count"),
    late_rate=("is_late", "mean"),
    median_delivery=("delivery_minutes", "median"),
    median_order_value=("order_value_inr", "median")
).reset_index()

decision_view["meets_5_percent_late_target"] = decision_view["late_rate"] <= 0.05
decision_view


Use this structure to write a decision memo:

**Decision:** What action is being considered?

**Evidence:** Which observed metrics support the decision?

**Uncertainty/limitations:** Is this a census of the defined population or a sample? Is the period representative? Are there missing or biased observations?

**Trade-offs/guardrails:** What else must be monitored (cost, cancellations, customer satisfaction, workload)?

**Next measurement:** What will you monitor after the action?


## 9. Final mastery challenge

Without copying the earlier explanations, write a short analysis for this scenario:

A retailer wants to know whether delivery performance in Bengaluru is meeting a 5% late-order target. Its warehouse table contains 50,000 order records for the month, but a pipeline incident may have dropped some events.

Your response must identify:
1. the unit of analysis and target population;
2. whether 50,000 rows automatically constitute a census;
3. the relevant parameter and the statistic you can calculate;
4. descriptive analyses you would perform;
5. what additional design/data is needed for valid inference;
6. data-engineering checks for the suspected dropped events;
7. a responsible business decision and monitoring plan.

**Mastery criterion:** Be precise about what the available data can and cannot establish. Do not claim causation from a simple comparison.

## 10. Completion checklist

- [ ] I can define statistics and explain its value in all three data roles.
- [ ] I can distinguish descriptive and inferential statistics.
- [ ] I can identify population, sample, parameter, statistic and census.
- [ ] I can explain sampling variability and why sample size does not remove bias.
- [ ] I can interpret mean, median, percentiles and standard deviation in context.
- [ ] I can classify analysis questions.
- [ ] I can design basic statistical data-quality monitoring checks.
- [ ] I completed the final mastery challenge in my own words.

**Next module prerequisite:** Be comfortable interpreting distributions, variation, sampling and uncertainty before moving to measures of central tendency, dispersion and probability.